In [0]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    TimestampType,
    DoubleType
)

In [0]:
source_file_path = (
    "/Volumes/online_retail/bronze/source_files/"
    "online_retail_2009-12.csv"
)

target_table = "online_retail.bronze.transactions_raw"

source_schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("source_sheet", StringType(), True),
    StructField("source_row_number", LongType(), True),
    StructField("batch_id", StringType(), True)
])

In [0]:
# read the source CSV
source_df = (
    spark.read
    .option("header", True)
    .schema(source_schema)
    .csv(source_file_path)
)

In [0]:
# prepare the bronze dataframe
bronze_df = (
    source_df
    .select(
        col("Invoice").alias("invoice"),
        col("StockCode").alias("stock_code"),
        col("Description").alias("description"),
        col("Quantity").alias("quantity"),
        col("InvoiceDate").alias("invoice_date"),
        col("Price").alias("price"),
        col("Customer ID").alias("customer_id"),
        col("Country").alias("country"),
        col("source_sheet"),
        col("source_row_number"),
        col("batch_id"),
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("source_file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

In [0]:
# validate before writing
bronze_df.printSchema()

root
 |-- invoice: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- source_row_number: long (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- source_file_path: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)



In [0]:
bronze_row_count = bronze_df.count()
print(f"Bronze rows before write: {bronze_row_count:,}")

Bronze rows before write: 45,228


In [0]:
bronze_df.show(5, truncate=False)

+-------+----------+-----------------------------------+--------+-------------------+-----+-----------+--------------+--------------+-----------------+--------+-------------------------+-------------------------------------------------------------------------+--------------------------+
|invoice|stock_code|description                        |quantity|invoice_date       |price|customer_id|country       |source_sheet  |source_row_number|batch_id|source_file_name         |source_file_path                                                         |ingestion_timestamp       |
+-------+----------+-----------------------------------+--------+-------------------+-----+-----------+--------------+--------------+-----------------+--------+-------------------------+-------------------------------------------------------------------------+--------------------------+
|489434 |85048     |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|Year 2009-2010|2  

In [0]:
# Write the managed Delta table
bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

In [0]:
# Validate the stored table
stored_bronze_df = spark.table(target_table)

stored_bronze_row_count = stored_bronze_df.count()

print(f"Stored Bronze rows: {stored_bronze_row_count:,}")

print("Row counts match:", stored_bronze_row_count == bronze_row_count)

Stored Bronze rows: 45,228
Row counts match: True
